# Inspect Agent4Rec Final Prompts

This notebook builds the current `Agent4RecYesNoScorer` prompt inputs for 10 MovieLens users.
It does not call an LLM. It only materializes the exact system/user messages that would be sent for the selected candidate groups.

The selected task must include train-only item rating statistics because Agent4Rec candidate prompts use `History ratings`.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from beyond_click_sim.scorers import Agent4RecProfileGenerator, Agent4RecYesNoScorer
from runners.in_distribution.interaction_prediction.methods.agent4rec_yes_no import (
    DATASET_CANDIDATE_COLUMNS,
    DATASET_COLUMN_LABELS,
)
from runners.in_distribution.interaction_prediction.methods.common import task_xy
from runners.in_distribution.interaction_prediction.task_builders import TASK_BUILDERS

In [ ]:
TASK_NAME = "ml-1m_item_stats_cap20_eval_users1000_cg5_m1_seed0"
N_USERS = 10

task = TASK_BUILDERS[TASK_NAME]()
dataset_name = str(task.manifest["dataset"])
candidate_group_column = task.schema.candidate_group_column
assert candidate_group_column is not None

xy = task_xy(task)
X_train, y_train = xy["train"]
X_test, y_test = xy["test"]

print(f"Task: {task.name}")
print(f"Train rows: {len(X_train):,}; test candidate rows: {len(X_test):,}")
print(f"Candidate columns: {DATASET_CANDIDATE_COLUMNS[dataset_name]}")

In [ ]:
scorer = Agent4RecYesNoScorer(
    client=None,
    model="prompt-preview-only",
    profile_generator=Agent4RecProfileGenerator(profile_components=("traits",)),
    candidate_description_columns=DATASET_CANDIDATE_COLUMNS[dataset_name],
    column_labels=DATASET_COLUMN_LABELS[dataset_name],
).fit(X_train, y_train)

scorer.profile_generator.manifest()

In [ ]:
selected_groups = []
seen_users = set()
for candidate_group, group in X_test.groupby(candidate_group_column, sort=False):
    user_id = group["user_id"].iloc[0]
    if user_id in seen_users:
        continue
    selected_groups.append((candidate_group, user_id, group.copy()))
    seen_users.add(user_id)
    if len(selected_groups) == N_USERS:
        break

assert len(selected_groups) == N_USERS, f"Expected {N_USERS} users, got {len(selected_groups)}"

In [ ]:
rows = []
examples = []
for position, (candidate_group, user_id, group) in enumerate(selected_groups, start=1):
    messages = scorer._build_messages(user_id=user_id, candidates=group)
    profile = scorer.profile_by_user_[user_id]
    examples.append(
        {
            "position": position,
            "user_id": user_id,
            "candidate_group": candidate_group,
            "group": group,
            "messages": messages,
            "profile": profile,
        }
    )
    rows.append(
        {
            "example": position,
            "user_id": user_id,
            "candidate_group": candidate_group,
            "candidate_rows": len(group),
            "positives": int(y_test.loc[group.index].sum()),
            "activity_group": profile.activity_group,
            "conformity_group": profile.conformity_group,
            "diversity_group": profile.diversity_group,
        }
    )

pd.DataFrame(rows)

In [ ]:
def code_block(text: str) -> str:
    return text.replace("```", "` ` `")

for example in examples:
    group = example["group"]
    messages = example["messages"]
    targets = y_test.loc[group.index].tolist()
    preview_columns = [
        "item_title",
        "item_rating_mean",
        "item_genres",
        "sampled",
    ]
    available_preview_columns = [column for column in preview_columns if column in group.columns]

    display(Markdown(f"## Example {example['position']}: user `{example['user_id']}` / group `{example['candidate_group']}`"))
    display(group[available_preview_columns].assign(target=targets))
    display(Markdown("### System prompt\n```text\n" + code_block(messages[0]["content"]) + "\n```"))
    display(Markdown("### User prompt\n```text\n" + code_block(messages[1]["content"]) + "\n```"))